# Choosing the default enrichment scorer

`tl.enrich` exposes decoupler's set scorers through `method=`: `ulm`, `mlm`, `ora`, `aucell`, `gsea`, `gsva`, `zscore`, `waggr` and `viper`. Each turns a set's features into one score in its own way, so the package has to pick one as the default. This case study picks it the way decoupler's own benchmark does: by how well each scorer recovers the perturbed source in a reference perturbation dataset.

The benchmark runs in decoupler's native domain, transcriptomics, not on morphology. KnockTF holds RNA-seq contrasts for single transcription-factor knockouts, and CollecTRI maps each factor to its target genes. A scorer that ranks the perturbed factor high from the expression contrast is the one to trust on feature sets in general. The default that `tl.enrich` carries is justified here, not demonstrated on image features.

In [ ]:
import os

# The benchmark's default parallelism trips numba's workqueue layer; one thread avoids it.
os.environ["NUMBA_NUM_THREADS"] = "1"

import decoupler as dc

adata = dc.ds.knocktf()
net = dc.op.collectri(organism="human")
print(adata)
f"{net['source'].nunique()} factors, {len(net)} edges in the network"

AnnData object with n_obs × n_vars = 388 × 21985
    obs: 'source', 'Species', 'Knock.Method', 'Biosample.Name', 'Profile.ID', 'Platform', 'TF.Class', 'TF.Superclass', 'Tissue.Type', 'Biosample.Type', 'Data.Source', 'Pubmed.ID', 'logFC', 'type_p'
    layers: None (.X)


'1185 factors, 42990 edges in the network'

Each row of `adata` is one knockout experiment, and `obs["source"]` names the factor it perturbed. `dc.bm.benchmark` scores every experiment with each method, then measures how well the method's ranking places the true factor first, as the area under the ROC curve (`auroc`) and the precision-recall curve (`auprc`).

In [ ]:
scored = ["ulm", "mlm", "aucell", "zscore"]
bench = dc.bm.benchmark(adata=adata, net=net, kws_decouple={"methods": scored, "tmin": 5})

table = (
    bench[bench["metric"].isin(["auroc", "auprc"])]
    .pivot_table(index="method", columns="metric", values="score")
    .sort_values("auroc", ascending=False)
)
table

metric,auprc,auroc
method,,
ulm,0.760249,0.719133
zscore,0.756472,0.717544
aucell,0.651371,0.705087
mlm,0.726610,0.687400


`ulm` leads on both measures, with `zscore` close behind and `aucell` and `mlm` lower. That is the standard decoupler result, and it is why `tl.enrich` defaults to `method="ulm"`: a linear model per set recovers the perturbed source as well as any scorer and better than most.

Four of the nine scorers stay out of this comparison. `ora` and `gsea` return unbounded scores, which the benchmark's area-under-curve metric does not accept, so they do not sit on the same scale as the rest. `gsva`, `viper` and `waggr` run but are far slower on a dataset this size, so they are left for a dedicated run rather than a quick default-picking pass.

## Summary

`tl.enrich` defaults to `method="ulm"` because a linear model per set recovers a known perturbation as well as any scorer decoupler's benchmark can score. The choice rests on a transcriptomic benchmark, decoupler's own domain, so it is a sensible default rather than a claim about morphology. On a given screen, scoring the same sets with a second, unrelated method such as `zscore` is the quick check that a call does not hang on the scorer.